# K-Nearest Neighbors for weather risk 

Multidimesnional K-nn classifier to determine if certain weather is safe/unsafe for planes to fly through

## Do these in order 
1. Balance dataset 
2. Training/testing split 
3. Normalizing 
4. Wrap KNN in KNeighborsClassifier(n_neighbors=5) ? maybe tune value of n later?  
5. Fit the model to the x_train and y_train ( these come from the normalized data)
6. Get/visualzie predictions(Can use MatPlotLib for this) - also print out accuracy of model to see if we cna fine tune 
7. (if we have time) - use predict_proba to convert the binary safe/unsafe tags into actual weather risk. 
8. Pass back this weather risk into calcuations.py file 

## 0. Imports

Uncomment or add any extra imports you need.

In [3]:
%pip install numpy pandas scikit-learn
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## 1. Configuration and loading data

**TODO:** Set `CSV_PATH` to balanced or raw dataset. Run `ml_dataset_scripts.py` from this folder first if you need to inspect counts; export a balanced CSV.

In [4]:
# Path relative to this notebook (usually backend/calculations)
CSV_PATH = Path("ml_ready_dataset.csv")
#print(Path("ml_ready_dataset (1).csv"))


#If the file is missing, create a tiny synthetic dataset 
if not CSV_PATH.is_file():
    print(f"Warning: {CSV_PATH} not found. Using synthetic data for pipeline practice.")
    rng = np.random.default_rng(42)
    n = 50000
    df = pd.DataFrame(
        {
            "wind_speed": rng.lognormal(3, 0.5, n),
            "wind_gust": rng.lognormal(3.2, 0.5, n),
            "precip_inches": rng.exponential(0.1, n),
            "humidity_pct": rng.uniform(20, 100, n),
            "lightning_strikes_10mi": rng.poisson(2, n),
            "unsafe_weather": rng.choice([0, 1], n, p=[0.85, 0.15]),
        }
    )
else:

    df = pd.read_csv(CSV_PATH)


print(df.shape)
print(df.dtypes)
df.head()

(4475051, 23)
max_temp_f            float64
min_temp_f            float64
temp_range_f          float64
max_dewpoint_f        float64
min_dewpoint_f        float64
dewpoint_range_f      float64
precip_in             float64
avg_wind_speed_kts    float64
snow_in               float64
avg_feel              float64
icing_risk              int64
has_snow                int64
has_precip              int64
high_wind               int64
below_freezing          int64
MONTH                   int64
DEP_1hrpre_num        float64
DEP_1hrpost_num       float64
unsafe_weather          int64
FL_DATE                object
ORIGIN                 object
DEST                   object
MKT_CARRIER            object
dtype: object


,max_temp_f,min_temp_f,temp_range_f,max_dewpoint_f,min_dewpoint_f,dewpoint_range_f,precip_in,avg_wind_speed_kts,snow_in,avg_feel,...,high_wind,below_freezing,MONTH,DEP_1hrpre_num,DEP_1hrpost_num,unsafe_weather,FL_DATE,ORIGIN,DEST,MKT_CARRIER
0,27.0,22.0,5.0,21.9,19.4,2.5,0.0200,3.128315,0.1000,19.545850,...,0,1,1,14.0,24.0,0,2023-01-02,MSP,SRQ,G4
1,45.0,34.0,11.0,45.0,32.0,13.0,0.4800,4.084188,0.0000,37.493580,...,0,0,1,26.0,9.0,0,2023-01-03,BOS,SRQ,G4
2,30.0,18.0,12.0,27.0,15.1,11.9,0.0100,8.429070,0.2000,14.716321,...,0,1,1,22.0,10.0,0,2023-01-05,MSP,SRQ,G4
3,43.0,28.0,15.0,30.0,21.9,8.1,0.0001,8.255275,0.0001,28.594500,...,0,1,1,18.0,22.0,0,2023-01-09,BOS,SRQ,G4
4,44.0,29.0,15.0,26.1,21.0,5.1,0.0000,7.125605,0.0000,27.086977,...,0,1,1,8.0,1.0,0,2023-01-09,IND,SRQ,G4


### Balance Dataset

### Split Data

In [27]:
# Split data here for training/testing - 80 -20 split 

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

data = df.drop(columns=['FL_DATE', 'ORIGIN', 'DEST', 'MKT_CARRIER'], errors='ignore')

# change to columns we want to use to train the data 
X = data.drop(columns=['unsafe_weather']) 

# change to the unsafe_weather/safe weather 
y = data['unsafe_weather']   


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) # tune later, but for now it is 80-20

print(X_test[0:5])
print(y)

         max_temp_f  min_temp_f  temp_range_f  max_dewpoint_f  min_dewpoint_f  \
3067041        88.0        72.0          16.0            72.0            62.0   
699754         42.0        33.0           9.0            33.1            23.0   
1532316        51.0        35.0          16.0            33.8            24.1   
425968         64.0        36.0          28.0            33.1            23.0   
3903645        76.0        56.0          20.0            52.0            43.0   

         dewpoint_range_f  precip_in  avg_wind_speed_kts  snow_in   avg_feel  \
3067041         10.000000        0.0            7.125605      0.0  79.258660   
699754          10.099998        0.0            7.560093      0.0  30.476420   
1532316          9.699999        0.0            4.344881      0.0  39.015950   
425968          10.099998        0.0            9.732533      0.0  49.440025   
3903645          9.000000        0.0           10.340817      0.0  65.164000   

         icing_risk  has_snow  h

### Normalize Data


In [20]:
# Adding normalizing 

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

data_normalized  = (scaler.fit_transform(data)) # All column data is now normalized. 

print("max:" , scaler.data_max_)
print(data_normalized)


max: [119.       97.       53.       84.       78.1      61.       22.5
  26.67757  11.5     105.18889   1.        1.        1.        1.
   1.       11.      107.      107.        1.     ]
[[0.21367521 0.31818182 0.07692308 ... 0.13084112 0.22429907 0.        ]
 [0.36752137 0.42727273 0.19230769 ... 0.24299065 0.08411215 0.        ]
 [0.23931624 0.28181818 0.21153846 ... 0.20560748 0.09345794 0.        ]
 ...
 [0.48717949 0.45454545 0.40384615 ... 0.03738318 0.06542056 0.        ]
 [0.61538462 0.48181818 0.63461538 ... 0.03738318 0.06542056 0.        ]
 [0.55555556 0.51818182 0.42307692 ... 0.09345794 0.03738318 0.        ]]


In [28]:
print(X_test.iloc[0])

max_temp_f            88.000000
min_temp_f            72.000000
temp_range_f          16.000000
max_dewpoint_f        72.000000
min_dewpoint_f        62.000000
dewpoint_range_f      10.000000
precip_in              0.000000
avg_wind_speed_kts     7.125605
snow_in                0.000000
avg_feel              79.258660
icing_risk             0.000000
has_snow               0.000000
has_precip             0.000000
high_wind              0.000000
below_freezing         0.000000
MONTH                  9.000000
DEP_1hrpre_num         7.000000
DEP_1hrpost_num       36.000000
Name: 3067041, dtype: float64


### KNN classifier 

In [34]:
#Wrap KNN in KNeighborsClassifier(n_neighbors=5) ?
from sklearn.neighbors import KNeighborsClassifier
neigh = KNeighborsClassifier(n_neighbors=5) # May tune later 

neigh.fit(X_train, y_train) # Uncomment and use Pandas to extract all weather data(features) as X and Safe/unsafe weather column as Y 

for i in range(0, 99):
     print(neigh.predict([X_test.iloc[i]]), y_test.iloc[i])

#print(neigh.predict([X_test.iloc[0:99]])) # Fill in weather data to predict (for testing )

#print(y_test.iloc[i]) # Compare to actual value of safe/unsafe weather for that data point.

c:\Users\miasa\OneDrive\Desktop\teamtech25-26\backend\venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(
c:\Users\miasa\OneDrive\Desktop\teamtech25-26\backend\venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(
c:\Users\miasa\OneDrive\Desktop\teamtech25-26\backend\venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(
c:\Users\miasa\OneDrive\Desktop\teamtech25-26\backend\venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(
c:\Users\miasa\OneDrive\Desktop\teamtech25-26\backend\venv\Lib\site-

[0] 0
[0] 0
[0] 0
[0] 0
[0] 0


c:\Users\miasa\OneDrive\Desktop\teamtech25-26\backend\venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(
c:\Users\miasa\OneDrive\Desktop\teamtech25-26\backend\venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(
c:\Users\miasa\OneDrive\Desktop\teamtech25-26\backend\venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(
c:\Users\miasa\OneDrive\Desktop\teamtech25-26\backend\venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(
c:\Users\miasa\OneDrive\Desktop\teamtech25-26\backend\venv\Lib\site-

[0] 0
[0] 0
[0] 0
[0] 0
[0] 0


c:\Users\miasa\OneDrive\Desktop\teamtech25-26\backend\venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(
c:\Users\miasa\OneDrive\Desktop\teamtech25-26\backend\venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(
c:\Users\miasa\OneDrive\Desktop\teamtech25-26\backend\venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(
c:\Users\miasa\OneDrive\Desktop\teamtech25-26\backend\venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(
c:\Users\miasa\OneDrive\Desktop\teamtech25-26\backend\venv\Lib\site-

...

c:\Users\miasa\OneDrive\Desktop\teamtech25-26\backend\venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(
c:\Users\miasa\OneDrive\Desktop\teamtech25-26\backend\venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(
c:\Users\miasa\OneDrive\Desktop\teamtech25-26\backend\venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(
c:\Users\miasa\OneDrive\Desktop\teamtech25-26\backend\venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(


[0] 0
[0] 0
[0] 0
[0] 0


c:\Users\miasa\OneDrive\Desktop\teamtech25-26\backend\venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(
c:\Users\miasa\OneDrive\Desktop\teamtech25-26\backend\venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(
c:\Users\miasa\OneDrive\Desktop\teamtech25-26\backend\venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(
c:\Users\miasa\OneDrive\Desktop\teamtech25-26\backend\venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(


[0] 0
[0] 0
[0] 0


c:\Users\miasa\OneDrive\Desktop\teamtech25-26\backend\venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(
c:\Users\miasa\OneDrive\Desktop\teamtech25-26\backend\venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(


##### Visualization of Accuracy/Confusion Matrix to tune above


In [ ]:
# 